# Matrix Multiplication

**Goal:** Implement matrix multiplication from scratch in PyTorch, validate against `torch.matmul`, and develop intuition for shapes, cost, and batched/einsum forms.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()  # reads config.toml -> device/seed/dtype (mps on Apple Silicon)
print("running on:", device)

running on: mps


## From Scratch: Triple-Loop Matrix Multiplication

The definition `C[i, j] = sum_k A[i, k] * B[k, j]` implemented explicitly.  
We use small tensors (CPU scalars) to keep the Python loop fast enough for demonstration.

In [2]:
def matmul_triple_loop(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Dense matmul via explicit triple loop — O(m*n*p).

    Args:
        A: Shape (m, n).
        B: Shape (n, p).

    Returns:
        C: Shape (m, p).
    """
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"Inner dimensions must match: {n} != {n2}"

    # Work on CPU for the loop; move back to original device at the end.
    A_cpu = A.cpu().float()
    B_cpu = B.cpu().float()
    C_cpu = torch.zeros(m, p)

    for i in range(m):
        for k in range(n):
            for j in range(p):
                C_cpu[i, j] = C_cpu[i, j] + A_cpu[i, k] * B_cpu[k, j]

    return C_cpu.to(A.device)


# Small example
A_small = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], device=device)
B_small = torch.tensor([[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]], device=device)

C_loop = matmul_triple_loop(A_small, B_small)
print("A shape:", A_small.shape)
print("B shape:", B_small.shape)
print("C shape:", C_loop.shape)
print("C (loop):\n", C_loop)

A shape: torch.Size([3, 2])
B shape: torch.Size([2, 3])
C shape: torch.Size([3, 3])
C (loop):
 

tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]], device='mps:0')


## Vectorised Matrix Multiplication

Replace the inner two loops with a vectorised dot-product over rows x columns.

In [3]:
def matmul_vectorised(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Vectorised matmul: row of A dot-producted with each column of B.

    Args:
        A: Shape (m, n).
        B: Shape (n, p).

    Returns:
        C: Shape (m, p).
    """
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"Inner dimensions must match: {n} != {n2}"

    C = torch.zeros(m, p, device=A.device, dtype=A.dtype)
    for i in range(m):
        # B.T has shape (p, n); B.T @ A[i] -> (p,)
        C[i] = B.T @ A[i]
    return C


C_vec = matmul_vectorised(A_small, B_small)
print("C (vectorised):\n", C_vec)

C (vectorised):
 tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]], device='mps:0')


## Validation Against `torch.matmul`

In [4]:
C_ref = torch.matmul(A_small, B_small)

assert torch.allclose(C_loop.to(device), C_ref, atol=1e-5), "Triple-loop does not match torch.matmul!"
assert torch.allclose(C_vec, C_ref, atol=1e-5), "Vectorised does not match torch.matmul!"
print("Both from-scratch implementations match torch.matmul ✓")
print("C (reference):\n", C_ref)

Both from-scratch implementations match torch.matmul ✓
C (reference):
 tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]], device='mps:0')


## Idiomatic PyTorch: `@` Operator and `torch.einsum`

In [5]:
# Standard operator — calls optimised BLAS / Metal kernel
C_op = A_small @ B_small
print("@ operator result matches:", torch.allclose(C_op, C_ref, atol=1e-5))

# torch.einsum: explicit index notation
C_einsum = torch.einsum("ij,jk->ik", A_small, B_small)
print("einsum result matches:", torch.allclose(C_einsum, C_ref, atol=1e-5))

print("\nAll three forms agree ✓")

@ operator result matches: True
einsum result matches: True

All three forms agree ✓


## Batched Matrix Multiplication

`torch.bmm` and `@` broadcast over batch dimensions — crucial for attention score computation  
`QK^T` where both tensors have shape `(batch, heads, tokens, d_head)`.

In [6]:
# Simulate batched attention-like matmul
batch, heads, tokens, d_head = 2, 4, 8, 16
Q = torch.randn(batch, heads, tokens, d_head, device=device)
K = torch.randn(batch, heads, tokens, d_head, device=device)

# Score matrix: (batch, heads, tokens, tokens)
scores_bmm = torch.bmm(
    Q.reshape(batch * heads, tokens, d_head),
    K.reshape(batch * heads, tokens, d_head).transpose(-1, -2),
).reshape(batch, heads, tokens, tokens)

scores_at = Q @ K.transpose(-1, -2)  # Broadcast matmul via @

print("scores_bmm shape:", scores_bmm.shape)
print("scores_at  shape:", scores_at.shape)
assert torch.allclose(scores_bmm, scores_at, atol=1e-4), "Batched matmul mismatch!"
print("Batched forms agree ✓")

scores_bmm shape: torch.Size([2, 4, 8, 8])
scores_at  shape: torch.Size([2, 4, 8, 8])


Batched forms agree ✓


## Shape Compatibility: What Happens When Dimensions Don't Align

In [7]:
# Demonstrate shape constraint
X = torch.randn(3, 4, device=device)
Y = torch.randn(5, 6, device=device)  # inner dim 4 != 5 -> error

try:
    _ = X @ Y
except RuntimeError as e:
    print("Expected shape error:", e)

# Correct: inner dims match
Y_ok = torch.randn(4, 6, device=device)
Z = X @ Y_ok
print("\nValid product shape:", Z.shape, "  (3x4 @ 4x6 = 3x6)")

Expected shape error: mat1 and mat2 shapes cannot be multiplied (3x4 and 5x6)

Valid product shape: torch.Size([3, 6])   (3x4 @ 4x6 = 3x6)


## Takeaways

- **Shape rule:** `(m, n) @ (n, p) -> (m, p)`. Inner dims must match; outer dims define the output.
- **Cost:** O(m·n·p) multiply-accumulates (MACs). A 4096×4096 dense matmul performs ~69 billion MACs ≈ ~137 billion FLOPs (each MAC = 1 multiply + 1 add).
- **Non-commutativity:** `AB != BA` in general; order encodes *which transformation applies first*.
- **Batch matmul:** `@` broadcasts over leading dimensions, enabling vectorised attention over all heads/examples at once.
- **`einsum`:** expressive shorthand for contractions; explicit index labels help catch transposition bugs.
- **Always use `@` / `torch.matmul` in production** — they dispatch to BLAS/cuBLAS/Metal and are orders of magnitude faster than any Python loop.